# A limit is not a receipt (CODA-OR)

Filling in descent, verifier, and terminalizer names is still not standing if the support sieve is empty (OR-D3, OR-T5). A leftover measured in $\ell^2$ is not the leftover this classifier uses (OR-D2, Section 6A).

Run from the package root, or from this `jupyter_notebooks/` folder.

In [1]:
from pathlib import Path
import sys

p = Path.cwd().resolve()
if p.name == "jupyter_notebooks":
    root = p.parent
elif (p / "code" / "coda_or_core.py").is_file():
    root = p
elif (p / "coda_atlas" / "F005_coda" / "or" / "code").is_dir():
    root = p / "coda_atlas" / "F005_coda" / "or"
else:
    root = p

assert (root / "code").is_dir(), "run from package root or jupyter_notebooks/"
if str(root / "code") not in sys.path:
    sys.path.insert(0, str(root / "code"))

from sympy import Matrix, Rational
from coda_or_core import (
    ComparisonObject,
    EnrichedEnvironment,
    SupportArrow,
    SupportSieve,
    classify_diagram_square,
    evaluate_comparison_authority,
    R_EMPTY_SUPPORT,
    R_NO_TERMINALIZER,
    R_UNSUPPORTED_COMPONENT,
)
print("CODA-OR core loaded successfully.")

CODA-OR core loaded successfully.


## 1. Names Without Support and Refusal Codes (OR-T5–OR-T7)

The calculator checks support first. Filled names without support return `R_empty_support`. Missing terminalizers return `R_no_terminalizer`. Unsupported components return `R_unsupported_component`.

In [2]:
# Case A: Empty support sieve
named_empty = ComparisonObject(
    object_id="limit_L",
    weight_class="finite",
    support_sieve=SupportSieve(target_object_id="limit_L", arrows=()),
    descent_d="D",
    verifier_v="V",
    terminalizer_tau="tau_L",
)
res_empty = evaluate_comparison_authority(named_empty)
print("Empty support refusal:", res_empty.refusal_code)
assert res_empty.refusal_code == R_EMPTY_SUPPORT
assert res_empty.inhabited is False

# Case B: Missing local terminalizer
sieve_ok = SupportSieve(
    target_object_id="limit_L",
    arrows=(
        SupportArrow("A", "limit_L", True, "chi_A"),
    ),
)
named_no_term = ComparisonObject(
    object_id="limit_L",
    weight_class="finite",
    support_sieve=sieve_ok,
    descent_d="D",
    verifier_v="V",
    terminalizer_tau=None,
)
res_no_term = evaluate_comparison_authority(named_no_term)
print("Missing terminalizer refusal:", res_no_term.refusal_code)
assert res_no_term.refusal_code == R_NO_TERMINALIZER
assert res_no_term.inhabited is False

# Case C: Mixed components with uncertified source
sieve_mixed = SupportSieve(
    target_object_id="limit_L",
    arrows=(
        SupportArrow("A1", "limit_L", True, "chi_A1"),
        SupportArrow("A2", "limit_L", False, "chi_A2"),
    ),
)
named_mixed = ComparisonObject(
    object_id="limit_L",
    weight_class="finite",
    support_sieve=sieve_mixed,
    descent_d="D",
    verifier_v="V",
    terminalizer_tau="tau_L",
)
res_mixed = evaluate_comparison_authority(named_mixed, require_all_sources=True)
print("Unsupported component refusal:", res_mixed.refusal_code)
assert res_mixed.refusal_code == R_UNSUPPORTED_COMPONENT
assert res_mixed.inhabited is False

# Case D: Certified comparison object
filled = ComparisonObject(
    object_id="limit_L",
    weight_class="finite",
    support_sieve=sieve_ok,
    descent_d="D",
    verifier_v="V",
    terminalizer_tau="tau_L",
)
ok = evaluate_comparison_authority(filled)
print("With declared certified arrow inhabited:", ok.inhabited)
assert ok.inhabited is True
assert ok.authority_effect == "None"

Empty support refusal: R_empty_support
Missing terminalizer refusal: R_no_terminalizer
Unsupported component refusal: R_unsupported_component
With declared certified arrow inhabited: True


## 2. $\ell^1$ Leftover vs Least-Squares $\ell^2$ Residual

Repair column $(1,1,1)^T$ and defect $(0,0,10)^T$: the exact $\ell^1$ quotient leftover is $10$, while the least-squares Euclidean projection squared residual is $200/3$. The two metrics disagree.

In [3]:
env = EnrichedEnvironment(
    dim_c0=1,
    dim_c1=3,
    dim_c2=1,
    d0=Matrix([[1], [1], [1]]),
    d1=Matrix([[0, 0, 0]]),
)
row = classify_diagram_square(Matrix([[0], [0], [10]]), env)
print("status:", row.status.value)
print("L1 leftover:", row.quotient_distance)

repair = env.d0
vector = Matrix([[0], [0], [10]])
gram = repair.T * repair
coeff = gram.inv() * (repair.T * vector)
residual = vector - repair * coeff
l2sq = sum((entry ** 2 for entry in residual), start=Rational(0))
print("L2 squared residual:", l2sq)
print("same number?", row.quotient_distance == l2sq)

# Assertions
assert row.quotient_distance == Rational(10, 1)
assert l2sq == Rational(200, 3)
assert row.quotient_distance != l2sq
print("L1 vs L2 residual distinction verified.")

status: H^1DiagramTear
L1 leftover: 10
L2 squared residual: 200/3
same number? False
L1 vs L2 residual distinction verified.
